In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.ndimage import gaussian_filter
import re

from pybedtools import BedTool
from sklearn.cluster import KMeans

import subprocess

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai',  'Delayed', 'Fail1','Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

plot_order = ['Start', 'Delayed', 'Fail1', 'Sendai', 'Inter1', 'Inter2', 'Fail2', 'IPS']

In [ ]:
genes_path = '/u/project/cluo/terencew/reference/hg38_igvf/bed/genes/gencode.v43.autosome.dedup.names.bed'
genes = pd.read_csv(genes_path, sep='\t', header=None)
genes.index = [f'{x}_{y}' for x,y in zip(genes[0], genes[1])]
genes['chr_num'] = [int(x.split('chr')[1]) for x in genes[0]]
genes = genes.sort_values(by=['chr_num', 1]).iloc[:,:4]
genes.shape

### load enhancer methylation

In [ ]:
peaks_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/atac/peaks/chromhmm/justin.chromhmm.dedup.bed.gz'
peaks = pd.read_csv(peaks_path, sep='\t', header=None)
peaks.index = [f'{x}_{y}' for x,y in zip(peaks[0], peaks[1])]
peaks.shape

In [ ]:
peaks.head()

### link peaks to genes just using bedtools nearest

In [ ]:
def process_bed(s):
    peaks_mc = pd.read_csv(f'{projdir}/bed/features/rna_cluster/{s}.chromhmm_peaks_CGN-Both.full.bed.gz', sep='\t', header=None)
    peaks_mc.index = [f'{x}_{y}' for x,y in zip(peaks_mc[0], peaks_mc[1])]
    return peaks_mc

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_bed, tmp_order), total=len(tmp_order)))

In [ ]:
results[0].head()

In [ ]:
results[0].shape

In [ ]:
mc_results = [x[15] / x[16] for x in results]
peaks_df = pd.concat(mc_results, axis=1)
peaks_df.columns = tmp_order

for s in tmp_order:
    peaks_df[s].fillna(peaks_df[s].mean())
    
peaks_df = peaks_df[~peaks_df.index.duplicated()]
peaks_df.shape

In [ ]:
peaks_df.head()

In [ ]:
peak_types = peaks[10].unique()
peak_types.shape

In [ ]:
annots = peaks[14].unique()
annots

In [ ]:
a = 'Enhancer'
annot_peaks = peaks[peaks[14] == a]
annot_peaks = annot_peaks[~annot_peaks.index.duplicated()]
annot_peaks.shape

In [ ]:
annot_peaks.head()

In [ ]:
enh_df = peaks_df.reindex(annot_peaks.index)

In [ ]:
prod_traj = ['Start', 'Sendai', 'Inter1', 'Inter2', 'IPS'] 

s1 = prod_traj[0]
prod_diff = pd.DataFrame(index=enh_df.index)
for s2 in prod_traj[1:]:
    diff = (peaks_df[s2] - peaks_df[s1])
    tmp_str = f'{s1}_{s2}'
    prod_diff[tmp_str] = diff
    s1 = s2
    
prod_diff['Start_Inter1'] = peaks_df['Inter1'] - peaks_df['Start']
prod_diff.shape

In [ ]:
prod_diff.head()

### fail1 traj

In [ ]:
traj = ['Start', 'Delayed', 'Fail1']

s1 = traj[0]
fail_diff = pd.DataFrame(index=enh_df.index)
for s2 in traj[1:]:
    diff = (peaks_df[s2] - peaks_df[s1])
    tmp_str = f'{s1}_{s2}'
    fail_diff[tmp_str] = diff
    s1 = s2
    
fail_diff.shape

In [ ]:
fail_diff.head()

### look at how enhancers change over prod trajectory

In [ ]:
thresh = 0.10
s = 'Start_Sendai'
mask = prod_diff[s] > thresh
pos_enh = prod_diff[mask]
mask = prod_diff[s] < -thresh
neg_enh = prod_diff[mask]

pos_enh = pos_enh.dropna()
neg_enh = neg_enh.dropna()

pos_enh.shape, neg_enh.shape

In [ ]:
pos_enh.head()

In [ ]:
neg_enh.head()

In [ ]:
k = 6

In [ ]:
kmeans = KMeans(n_clusters=k)
kmeans.fit(pos_enh)
k_clusters = kmeans.predict(pos_enh)
pos_enh['cluster'] = k_clusters
pos_enh = pos_enh.sort_values(by=['cluster']).iloc[:,:4]
sns.heatmap(pos_enh, cmap='coolwarm', vmin=-0.5, vmax=0.5)

In [ ]:
kmeans = KMeans(n_clusters=k)
kmeans.fit(neg_enh)
k_clusters = kmeans.predict(neg_enh)
neg_enh['cluster'] = k_clusters
neg_enh = neg_enh.sort_values(by=['cluster']).iloc[:,:4]
sns.heatmap(neg_enh, cmap='coolwarm', vmin=-0.5, vmax=0.5)

In [ ]:
tmp_enh = peaks_df.loc[pos_enh.index][plot_order]
tmp_enh = tmp_enh.dropna()
kmeans = KMeans(n_clusters=k)
kmeans.fit(tmp_enh)
k_clusters = kmeans.predict(tmp_enh)
tmp_enh['cluster'] = k_clusters
tmp_enh = tmp_enh.sort_values(by=['cluster']).iloc[:,:-1]
sns.heatmap(tmp_enh, cmap='coolwarm', vmin=-0.5, vmax=0.5)

In [ ]:
tmp_enh = peaks_df.loc[neg_enh.index][plot_order]
tmp_enh = tmp_enh.dropna()
kmeans = KMeans(n_clusters=k)
kmeans.fit(tmp_enh)
k_clusters = kmeans.predict(tmp_enh)
tmp_enh['cluster'] = k_clusters
tmp_enh = tmp_enh.sort_values(by=['cluster']).iloc[:,:-1]
sns.heatmap(tmp_enh, cmap='coolwarm', vmin=0.2, vmax=0.8)

### fail diff

In [ ]:
thresh = 0.10
s = 'Start_Delayed'
mask = fail_diff[s] > thresh
pos_enh = fail_diff[mask]
mask = fail_diff[s] < -thresh
neg_enh = fail_diff[mask]

pos_enh = pos_enh.dropna()
neg_enh = neg_enh.dropna()

pos_enh.shape, neg_enh.shape

In [ ]:
kmeans = KMeans(n_clusters=k)
kmeans.fit(pos_enh)
k_clusters = kmeans.predict(pos_enh)
pos_enh['cluster'] = k_clusters
pos_enh = pos_enh.sort_values(by=['cluster']).iloc[:,:-1]
sns.heatmap(pos_enh, cmap='coolwarm', vmin=-0.5, vmax=0.5)

In [ ]:
kmeans = KMeans(n_clusters=k)
kmeans.fit(neg_enh)
k_clusters = kmeans.predict(neg_enh)
neg_enh['cluster'] = k_clusters
neg_enh = neg_enh.sort_values(by=['cluster']).iloc[:,:-1]
sns.heatmap(neg_enh, cmap='coolwarm', vmin=-0.5, vmax=0.5)

In [ ]:
tmp_enh = peaks_df.loc[pos_enh.index][plot_order]
tmp_enh = tmp_enh.dropna()
kmeans = KMeans(n_clusters=k)
kmeans.fit(tmp_enh)
k_clusters = kmeans.predict(tmp_enh)
tmp_enh['cluster'] = k_clusters
tmp_enh = tmp_enh.sort_values(by=['cluster']).iloc[:,:-1]
sns.heatmap(tmp_enh, cmap='coolwarm', vmin=0, vmax=1)

In [ ]:
tmp_enh = peaks_df.loc[neg_enh.index][plot_order]
tmp_enh = tmp_enh.dropna()
kmeans = KMeans(n_clusters=k)
kmeans.fit(tmp_enh)
k_clusters = kmeans.predict(tmp_enh)
tmp_enh['cluster'] = k_clusters
tmp_enh = tmp_enh.sort_values(by=['cluster']).iloc[:,:-1]
sns.heatmap(tmp_enh, cmap='coolwarm', vmin=0, vmax=1)

### write out interesting enhancers to file

In [ ]:
thresh = 0.10
s = 'Start_Sendai'
mask = prod_diff[s] > thresh
prod_pos_enh = prod_diff[mask]
mask = prod_diff[s] < -thresh
prod_neg_enh = prod_diff[mask]

prod_pos_enh = prod_pos_enh.dropna()
prod_neg_enh = prod_neg_enh.dropna()

prod_pos_enh.shape, prod_neg_enh.shape

In [ ]:
thresh = 0.10
s = 'Start_Delayed'
mask = fail_diff[s] > thresh
fail_pos_enh = fail_diff[mask]
mask = fail_diff[s] < -thresh
fail_neg_enh = fail_diff[mask]

fail_pos_enh = fail_pos_enh.dropna()
fail_neg_enh = fail_neg_enh.dropna()

fail_pos_enh.shape, fail_neg_enh.shape

In [ ]:
mask = peaks.loc[prod_pos_enh.index][13] == 'Sendai'
prod_pos_enh = prod_pos_enh[mask]
mask = peaks.loc[prod_neg_enh.index][13] == 'Sendai'
prod_neg_enh = prod_neg_enh[mask]
mask = peaks.loc[fail_pos_enh.index][13] == 'Sendai'
fail_pos_enh = fail_pos_enh[mask]
mask = peaks.loc[fail_neg_enh.index][13] == 'Sendai'
fail_neg_enh = fail_neg_enh[mask]

In [ ]:
prod_pos_enh.shape, prod_neg_enh.shape, fail_pos_enh.shape, fail_neg_enh.shape

In [ ]:
prod_pos_enh.to_csv(f'{projdir}/csv/peaks/enhancers/prod_pos_enh.csv', sep='\t')
prod_neg_enh.to_csv(f'{projdir}/csv/peaks/enhancers/prod_neg_enh.csv', sep='\t')
fail_pos_enh.to_csv(f'{projdir}/csv/peaks/enhancers/fail_pos_enh.csv', sep='\t')
fail_neg_enh.to_csv(f'{projdir}/csv/peaks/enhancers/fail_neg_enh.csv', sep='\t')

In [ ]:
peaks_df.to_csv(f'{projdir}/csv/peaks/enhancers/enhancer_mcg.csv', sep='\t')

### write enhancers to bed file

In [ ]:
prod_pos_enh.to_csv(f'{projdir}/csv/peaks/enhancers/prod_pos_enh.csv', sep='\t')
prod_neg_enh.to_csv(f'{projdir}/csv/peaks/enhancers/prod_neg_enh.csv', sep='\t')
fail_pos_enh.to_csv(f'{projdir}/csv/peaks/enhancers/fail_pos_enh.csv', sep='\t')
fail_neg_enh.to_csv(f'{projdir}/csv/peaks/enhancers/fail_neg_enh.csv', sep='\t')

In [ ]:
tmp_peaks = peaks.loc[prod_neg_enh.index]
out = f'{projdir}/csv/peaks/enhancers/prod_neg_enh.bed'
tmp_peaks.to_csv(out, sep='\t', header=None, index=None)

subprocess.run(['bgzip', '-f', out], check=True)
subprocess.run(['tabix', '-p', 'bed', f'{out}.gz'], check=True)
tmp_peaks.to_csv(out, sep='\t', header=None, index=None)

In [ ]:
tmp_peaks = peaks.loc[fail_pos_enh.index]
out = f'{projdir}/csv/peaks/enhancers/fail_pos_enh.bed'
tmp_peaks.to_csv(out, sep='\t', header=None, index=None)

subprocess.run(['bgzip', '-f', out], check=True)
subprocess.run(['tabix', '-p', 'bed', f'{out}.gz'], check=True)
tmp_peaks.to_csv(out, sep='\t', header=None, index=None)

In [ ]:
tmp_peaks = peaks.loc[fail_neg_enh.index]
out = f'{projdir}/csv/peaks/enhancers/fail_neg_enh.bed'
tmp_peaks.to_csv(out, sep='\t', header=None, index=None)

subprocess.run(['bgzip', '-f', out], check=True)
subprocess.run(['tabix', '-p', 'bed', f'{out}.gz'], check=True)
tmp_peaks.to_csv(out, sep='\t', header=None, index=None)

In [ ]:
!date